# PET Reconstruction | Portfolio edition

Coursework from **EEN180, Chalmers University of Technology**. All code was developed jointly in a pair, including Neo Barrdahl ([Nero-Vanetti](https://github.com/Nero-Vanetti)). The collaborator's name is not listed in this edition.

Start with the [project README](README.md) for the methods, selected results and setup. See the [review notes](../docs/REVIEW_NOTES.md) for implementation limitations and the small packaging changes made to this copy.

The original assignment structure, written answers and saved results are retained below. These are historical coursework outputs, not a claim that every scientific conclusion has been independently validated. Put `PETdata.mat` beside this notebook before running it.


# EEN180: PET Image Reconstruction

Modern medical imaging is generally using iterative reconstruction methods. These methods use the forward model of the imaging process (object to projection, object to $k$-space etc.) instead of the opposite that is used by analytical reconstruction methods. The forward model is often easier to describe and better known. Iterative reconstruction methods are typically characterized by a reduced noise sensitivity and enables straight-forward ways of including features like prior knowledge or sparsity constraints.

The objective of the project is to reconstruct PET images from sinogram data using iterative reconstruction. These instructions assume that MATLAB is used as the programming environment, but Python is also allowed.

## Notes

- The project has a total of 10 pts. To pass, you need  5 pts.
- This is a group exercise. Discussions between groups are encouraged while copying of code among groups is strictly prohibited. Any evidence of plagiarism will be dealt with according to Chalmers regulations by the teaching team.
- The objective of this assignment is algorithm implementation. You are allowed to use some general purpose functions such as fft and ifft. Specific functions that are directly related to the assignment such as  open-source code for iterative PET/SPECT reconstruction cannot be used, except for your own benchmark of your code.

## Theory
The reconstruction method used in this project builds on the principle that the forward projection (object to sinogram) can be described as a linear process $g=Af$ (for each 2D slice). The vector $g$ equals a flattened version of the sinogram matrix $G$ while the vector $f$ equals a flattened version of the image matrix $F$.

The matrix $A$ is often called the system matrix and is a transformation matrix that transforms image pixel values to projection (sinogram) data. The elements of $A$, $a_{ij}$, are thus sensitivity factors that characterize the sensitivity of each voxel to the collection of projection measurements:  represents the sensitivity (probability) that an annihilation event in pixel $j$ is measured in line $i$ (line of response, LOR). Many different effects contribute to these probabilities; geometry, object attenuation, individual detector efficiencies, etc. In this project, we only consider the geometry effect on the elements of $A$. Each pixel in the object (image) can contribute to a count in a certain LOR $i$, with different probabilities determined by the scanner geometry relative to the object. The total count in a LOR $i$ is then the total contribution from all  pixels:

$g_i = \sum_{j=1}^m a_{ij} f_j$

Thus, to calculate $A$ we need to determine the contribution of each pixel in the image to all LORs in the sinogram, i.e. how much of a specific pixel in the image "ends up" in every single possible LOR. 

## Data
### About the sinogram data
Load the data `'PETdata.mat'` to load the data needed for the project. The sinogram data is stored in `G` (noise free) and `G_noisy` (with noise). The size of the sinogram is 87 rows $\times$ 180 columns $\times$ 26 slices. The sinogram was formed based on multiple parallel beam projections of an object from 180 different directions: $0^\circ$ to $179^\circ$ with a step size of $1^\circ$. You will also be supplied a downsampled `G_small`, where we have downsampled the sinogram with a factor 2, in order to make the prototyping of your algorithms a bit faster.

### About the reconstructed images
Unlike other medical imaging modalities, nuclear medicine images typically have an inverted gray scale. This means that the images should be viewed such that the small pixel values are light (white) and large pixel values are dark (black), unless otherwise stated. This can easily be done in MATLAB or 3D slicer.

In this project, the size of the first dimension of the sinogram equals the size of the sides of the (square) images that are to be reconstructed, i.e. the size of the output images should be 87 $\times$ 87 $\times$ 26 voxels, where 26 represents the number of slices. Note that this is different from the CT project where the size of the sinogram was larger than the reconstructed image. All reconstructions should be made slice by slice. 


## Preparations

Load the data:

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.io import loadmat
from skimage.transform import radon, iradon

# Load data
data = loadmat('PETdata.mat')
G = data['G']
G_noisy = data['G_noisy']


# Print size
print(f'Size G:  bins x angles x slices = {G.shape[0]} x {G.shape[1]} x {G.shape[2]}')


dtheta = 180 / G.shape[1]
theta = np.arange(0, 180, dtheta)
image_size = np.array([G.shape[0], G.shape[0], G.shape[2]])

angles = np.arange(0, 180)  # angles for projections
slice_idx = 10  # single slice to reconstruct in the tasks below. Choose this as you wish.

# Prepare a smaller version of the data for faster computations
G_small = G[range(0, G.shape[0], 2), :, slice_idx]
G_small = G_small.reshape(*G_small.shape, 1)
image_size_small = np.array([G_small.shape[0], G_small.shape[0], 1])
print(f'Size G_small:  bins x angles x slices = {G_small.shape[0]} x {G_small.shape[1]} x {G_small.shape[2]}')


Size G:  bins x angles x slices = 87 x 180 x 26
Size G_small:  bins x angles x slices = 44 x 180 x 1


## Tasks

Using the given sinogram data together with the course material (P&L 9.2.3, Chappell 10.2) and/or the suggested reading, complete the following tasks:

## Part A: Forward projection

We know that the forward projection (object to sinogram) can be described as a linear process $g=Af$ using the flattened $G\rightarrow g$ and flattened $F\rightarrow f$ of each 2D slice. The size of the sinogram matrix $G$ equals $L\times\Theta$ and the size of the image matrix $F$ equals $R\times C$. In this project, we have that $L=R=C$ and that $\Theta = 180$. To reconstruct an image, we need to find the matrix $A$, and then solve for $f$. See P&L 9.2.3 for more details.

### A1 (0.5 pts)

What are the sizes of $g$ and $f$ (in unit pixels)? What should the size of $A$ be?

**Answer:**

For one 2D slice the image F has size 87x87 which gives the vector f a size of 7569 pixels. 

The sinogram G has 87x180 which gives the vector g a size of 15660 pixels.

Since the forward model is g=A*f, the system matrix A must have a size of 15660x7569

### A2 (1.5 pts)

Write a function `compute_forward_matrix` that returns the matrix $A$ given the projection angles, the size of the sinogram and the dimensions of the image.

Observe: In this project, we will use Matlab-style orientation of indexing and memory. This means that you need supply the argument `order='F'` (`'F'` means "Fortran") to a number of functions that create or reshape arrays, e.g. `np.zeros`, `np.reshape` and `np.ravel`.

Hint: You will find the scikit-image `radon` function helpful

Hint 2: Think about how Python indexes array elements versus the angle in degrees.

Hint 3: Use `np.ravel(x, order='F')` or the array method `x.ravel(order='F')` to flatten arrays.

In [2]:
def compute_forward_matrix(thetas, L, R, C):
    """
    Compute the forward projection matrix A.
    
    Parameters:
    -----------
    thetas : array
        Projection angles in degrees
    L : int
        Number of detector bins (sinogram rows)
    R : int
        Image rows
    C : int
        Image columns
    
    Returns:
    --------
    A : ndarray
        Forward projection matrix of shape [size: ...]
    """
    A = np.zeros((L * len(thetas), R * C), dtype=np.float32, order='F')
    
    for th_idx, th in enumerate(thetas):
        for j in range(R * C):
            # YOUR CODE HERE
            image = np.zeros((R, C), dtype=np.float32, order='F')
            image_flattened = image.ravel(order='F')
            image_flattened[j] = 1.0
            image = image_flattened.reshape((R, C), order='F')
            sinogram = radon(image, theta=[float(th)], circle=True)
            sinogram_flattened = sinogram.ravel(order='F')
            A[th_idx * L:(th_idx + 1) * L, j] = sinogram_flattened[:L]

    return A

### A3 (1.0 pts)

Write a function `NAIVE` that performs the naive reconstruction by solving the problem $g=Af$, i.e. solving for $f$.

In [3]:
def NAIVE(A, G, output_size):
    """
    Naive reconstruction by solving g = Af for f.
    
    Parameters:
    -----------
    A : ndarray
        Forward projection matrix [...]
    G : ndarray
        Sinogram data [height x width] or [height x width x slices]
    output_size : tuple
        Output image size [height, width, slices]
    
    Returns:
    --------
    F : ndarray
        Reconstructed image of shape output_size
    """
    F = np.zeros(output_size, dtype=np.float32, order='F')
    
    # Handle 2D case (single slice)
    for s in range(output_size[2]):
        # YOUR CODE HERE

        g = G[:, :, s].ravel(order='F') if G.ndim == 3 else G.ravel(order='F')
        f, _, _, _ = np.linalg.lstsq(A, g, rcond=None)
        F[:, :, s] = f.reshape((output_size[0], output_size[1]), order='F')
    
    return F

Apply your function to the noise free sinogram data to verify that the $A$ you computed in A2 is correct.

Note: This reconstruction is very slow, it is sufficient to reconstruct a single slice (e.g. slice 10). Make sure to save the slice for later use in task C1. First try your function on the resized `G_small` for prototyping (you should see a rough outline of a brain with a checkerboard pattern), and then move on to the bigger sinogram `G`. 

Tip: Save the $A$ matrix locally and load it to avoid recalculating it!

Visualize the reconstructed slice.

In [4]:
# Compute forward matrix for the small data!
A_small = compute_forward_matrix(theta, image_size_small[0], image_size_small[0], image_size_small[0])
print(f'Size A:  {A_small.shape[0]} x {A_small.shape[1]}')

# Perform naive reconstruction on a single slice
F_naive_small = NAIVE(A_small, G_small, np.array([image_size_small[0], image_size_small[0], 1]))

np.save('A_small.npy', A_small)
np.save('F_naive_small_slice.npy', F_naive_small[:, :, 0])

# Visualize
plt.figure(figsize=(8, 8))
plt.imshow(F_naive_small[:, :, 0], cmap='gray_r')
plt.title('Naive')
plt.axis('image')
plt.tight_layout()
plt.show()

  warn(


Size A:  7920 x 1936


![Saved output from cell 12, figure 3](../assets/notebook-results/pet-cell-12-figure-2.png)


In [7]:
# Compute forward matrix (this may take a while)
A = compute_forward_matrix(theta, image_size[0], image_size[0], image_size[0])
print(f'Size A:  {A.shape[0]} x {A.shape[1]}')
np.save('A.npy', A)
# Perform naive reconstruction on a single slice
F_naive = NAIVE(A, G[:, :, slice_idx], np.array([image_size[0], image_size[0], 1]))

np.save('A.npy', A)
np.save(f'F_naive_slice_{slice_idx}.npy', F_naive[:, :, 0])

# Visualize
plt.figure(figsize=(8, 8))
plt.imshow(F_naive[:, :, 0], cmap='gray_r')
plt.title('Naive')
plt.axis('image')
plt.tight_layout()
plt.show()

Size A:  15660 x 7569


![Saved output from cell 13, figure 2](../assets/notebook-results/pet-cell-13-figure-1.png)


## Part B: Iterative reconstruction

The reconstruction method used in modern PET is based on maximum-likelihood expectation maximization (MLEM) (For computation speed a method called ordered subset expectation maximization (OSEM) is used in practical implementations, but it is based on the same principle.). MLEM is a statistical method used to estimate unknown parameters, in this case the image, based on measured data and a probability distribution describing the measurement noise.

### B1 (1.0 pts)

For PET, the noise distribution is given by the Poisson distribution since the detector counts are related to radioactive decay, i.e.

$$P(g_i)=e^{-{\bar g_i}}\frac{{\bar g_i}^{g_i}}{g_i!},$$

where $g_i$ is the measured count for a given combination of position and angle, $i$, in the sinogram (i.e. a specific LOR) and where ${\bar g_i}$ is the average count for that LOR. Note that the index $i$ refers to one element in the sinogram, that is, it indexes the flattened vector $g$. The MLEM algorithm seeks to find the image $f$ that maximizes the logarithm of the Poisson likelihood objective function:

$$L(f)=P(g|f)=\prod_i P(g_i).$$

Derive the log-likelihood function based on the equations above. Observe that it is possible to write LaTeX in the notebook, using dollar-signs (\$).

**Answer**:

Using $\bar g_i = (Af)_i$ and $P(g_i)=e^{-\bar g_i}\frac{\bar g_i^{g_i}}{g_i!}$, the likelihood can be written as  
$L(f)=\prod_i e^{-(Af)_i}\frac{(Af)_i^{g_i}}{g_i!}$.

Taking the logarithm gives the log-likelihood  
$\ell(f)=\log L(f)=\sum_i \left[-(Af)_i + g_i \log (Af)_i - \log(g_i!)\right]$.

The term $\sum_i \log(g_i!)$ is constant with respect to $f$ and can be omitted when maximizing the log-likelihood.


### B2 (0.5 pts)

Iterative reconstruction schemes build on iteratively updating an estimate (guess) of the image to make it more and more aligned with the measured data.  What is an appropriate start image $f^{(0)}$?

**Answer:**

An appropriate start image $f^{(0)}$ is a uniform image with strictly positive values, for example an image where all pixels are set to 1. This ensures numerical stability and allows the MLEM updates to be well defined for all pixels.

### B3 (1.5 pts)

It can be shown that the following ML-EM iteration converges to the ML solution:

$$\bar{g}^{(k)} = Af^{(k)}$$

$$f_j^{(k+1)} = \frac{f_j^{(k)}}{\sum_i a_{ij}}\sum_i a_{ij}\frac{g_i}{\bar{g}_i^{(k)}},$$

where $k$ is the iteration number and the index $j$ refers to a given pixel in the reconstructed image.

- Write a function `ML_solution` that implements this solution for a given slice. 
- Write a function `sum_of_squares` that computes the sum-of-squares error between two arrays.
- Write a function `MLEM` that uses this iterative scheme to reconstruct an image. 

In [8]:
def ML_solution(F_k, A, G):
    """
    Compute the maximum likelihood solution for a single iteration.
    
    Parameters:
    -----------
    F_k : ndarray
        Current estimate of the image (flattened) [rows*columns]
    A : ndarray
        Forward projection matrix [...]
    G : ndarray
        Sinogram data (flattened) [bins*angles]
    
    Returns:
    --------
    F_new : ndarray
        Updated image estimate (flattened)
    """
    # YOUR CODE HERE
    eps = 1e-8
    g_bar = A @ F_k
    ratio = G / (g_bar + eps)
    backproj = A.T @ ratio
    sens = np.sum(A, axis=0)
    F_new = F_k * (backproj / (sens + eps))
    
    return F_new

def sum_of_squares(F, G):
    """
    Compute the objective function value.
    
    Parameters:
    -----------
    F_k : ndarray
        Current estimate of the image (flattened) [rows*columns*slices]
    G : ndarray
        Sinogram data (flattened) [bins*angles*slices]
    
    Returns:
    --------
    error : float
        Sum of squares error
    """
    # YOUR CODE HERE
    diff = F - G
    error = float(np.sum(diff * diff))
    return error
    

def MLEM(A, G, output_size, nbr_iter):
    """
    MLEM reconstruction algorithm.
    
    Parameters:
    -----------
    A : ndarray
        Forward projection matrix [...]
    G : ndarray
        Sinogram data (flattened) [bins*angles, slices]
    output_size : tuple
        Output image size [height, width, num_slices]
    nbr_iter : int
        Number of iterations
    
    Returns:
    --------
    F : ndarray
        Reconstructed images at each iteration, shape [height, width, num_slices, nbr_iter]
    r_error : ndarray
        Residual error at each iteration
    objective_function : ndarray
        Objective function value at each iteration
    """
    # YOUR CODE HERE
    if G.ndim == 3:
        G = G.reshape((G.shape[0] * G.shape[1], G.shape[2]), order='F')
    else:
        G = G.reshape((G.shape[0] * G.shape[1], 1), order='F')
    
    F = np.ones((output_size[0] * output_size[1], output_size[2], nbr_iter), dtype=np.float32, order='F')
    error = np.zeros(nbr_iter, dtype=np.float32)
    
    F_proj = np.zeros_like(G, dtype=np.float32, order='F')
    for k in range(nbr_iter):
        for s in range(output_size[2]):
            F_k = F[:, s, k-1] if k > 0 else F[:, s, 0]
            F[:, s, k] = ML_solution(F_k, A, G[:, s])
            F_proj[:, s] = A @ F[:, s, k]
        error[k] = sum_of_squares(G, F_proj)
    
    F = F.reshape((*output_size, nbr_iter), order='F')
    return F, error


You should compute the sum-of-squared difference between the sinogram and the forward projection at each iteration.

Apply your function to the noise free sinogram and use the sum-of-squared residuals to decide when the reconstruction algorithm has converged.

Note: Make sure to save your reconstructed image for later use in task C4.

Visualize a reconstructed slice for different iteration numbers and a plot of the sum-of-squared residuals vs iteration number.

In [9]:
iterations = 100
F_mlem, error = MLEM(A, G[:, :, slice_idx], np.array([image_size[0], image_size[0], 1]), iterations)

# YOUR CODE HERE
iters_to_show = [0, 1, 2, 5, 10, 20, 50, 99]
fig, axes = plt.subplots(1, len(iters_to_show), figsize=(16, 4))

for i, iter_idx in enumerate(iters_to_show):
    img_slice = F_mlem[:, :, 0, iter_idx]
    axes[i].imshow(img_slice, cmap='gray_r')
    axes[i].set_title(f'Iteration {iter_idx + 1}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 5))
plt.plot(range(1, iterations + 1), error)
plt.title('Convergence: Sum-of-Squared Residuals')
plt.xlabel('Iteration number')
plt.ylabel('Sum-of-Squared Error')
plt.grid(True)
plt.show()

final_reconstruction = F_mlem[:, :, :, -1]
np.save('F_mlem_final.npy', final_reconstruction)
print("Saved final reconstruction to 'F_mlem_final.npy'")

Saved final reconstruction to 'F_mlem_final.npy'


![Saved output from cell 22, figure 1](../assets/pet/noise-free-mlem.png)

![Saved output from cell 22, figure 2](../assets/notebook-results/pet-cell-22-figure-1.png)


## Part C: Iterative reconstruction with noisy data

### C1 (0.5 pts)

Apply the naive reconstruction from task A3 to the noisy sinogram, and compare the reconstructed noisy image with the noise free version. Visualize a slice of the reconstructed noisy image and comment on the results.

In [10]:
import os
F_naive_noisy = NAIVE(A, G_noisy[:, :, slice_idx], np.array([image_size[0], image_size[0], 1]))

# YOUR CODE HERE# Visualize the noisy reconstruction
if os.path.isfile('A.npy'):
    A = np.load('A.npy')
    
plt.figure(figsize=(8, 8))
plt.imshow(F_naive_noisy[:, :, 0], cmap='gray_r')
plt.title(f'Naive Reconstruction (Noisy Data) - Slice {slice_idx}')
plt.axis('off')
plt.colorbar(label='Intensity')
plt.show()

if 'F_naive' in locals():
    fig, ax = plt.subplots(1, 2, figsize=(12, 5))
    ax[0].imshow(F_naive[:, :, 0], cmap='gray_r')
    ax[0].set_title('Noise-free Naive')
    ax[0].axis('off')
    
    ax[1].imshow(F_naive_noisy[:, :, 0], cmap='gray_r')
    ax[1].set_title('Noisy Naive')
    ax[1].axis('off')
    plt.show()

![Saved output from cell 24, figure 1](../assets/notebook-results/pet-cell-24-figure-0.png)

![Saved output from cell 24, figure 2](../assets/notebook-results/pet-cell-24-figure-1.png)


**Answer**:



* **Noise-free Naive:** This image shows a recognizable, but blurry, reconstruction of the object. The internal structures and varying densities of the object are distinguishable.
* **Noisy Naive:** In stark contrast, the noisy reconstruction is almost entirely unrecognizable. The structural information seen in the noise-free version is completely overwhelmed by high-frequency granular noise. Only a very small and general sense of the orignial object remain in the central region's location but have almost no definable features.

This shows that naive reconstruction is highly susceptible to noise. This is probably because the naive method isn't able to handle the signal to noise ration.

### C2 (1.0 pts)

Write a function `FBP` that uses filtered back-projection (FBP) to reconstruct an image. Apply your function to the noisy and noise-free sinogram.

Hint: You will find the scikit-image `iradon` function helpful. Visualize a few slices of FBP reconstructed noisy and noise-free images and comment on the results.

In [11]:
def FBP(G, theta, output_size):
    """
    Filtered back-projection reconstruction.
    
    Parameters:
    -----------
    G : ndarray
        Sinogram data [bins x angles x slices]
    theta : array
        Projection angles
    output_size : tuple
        Output image size [height x width x num_slices]
    
    Returns:
    --------
    F : ndarray
        Reconstructed images
    """
    F = np.zeros(output_size, dtype=np.float32, order='F')
    
    img_side_length = output_size[0]

    for s in range(output_size[2]):
        # YOUR CODE HERE
        sinogram = G[:, :, s]
        reconstruction = iradon(sinogram, theta=theta, filter_name='ramp', circle=False, output_size=img_side_length)
        F[:, :, s] = reconstruction
    return F

In [12]:
# Apply FBP
F_fbp = FBP(G, angles, image_size)
F_fbp_noisy = FBP(G_noisy, angles, image_size)

# YOUR CODE HERE
slice_idx = 15

fig, ax = plt.subplots(1, 2, figsize=(12, 6))

ax[0].imshow(F_fbp[:, :, slice_idx], cmap='gray_r')
ax[0].set_title('FBP (Noise-free)')
ax[0].axis('off')

ax[1].imshow(F_fbp_noisy[:, :, slice_idx], cmap='gray_r')
ax[1].set_title('FBP (Noisy)')
ax[1].axis('off')

plt.tight_layout()
plt.show()

![Saved output from cell 29, figure 1](../assets/notebook-results/pet-cell-29-figure-0.png)


**Answer**:



FBP (Noise-free): The reconstruction is relativly sharp and structurally accurate. The characteristic fog seen in naive back-projection is removed, revealing more distinct internal features of the brain phantom. While the image is improved it still isn't very detailed.

FBP (Noisy): The object is successfully recovered and clearly visible, which is a massive improvement over the naive method. The image does however still suffer from significant graininess.

Conclusion FBP effectively restores image sharpness and contrast compared to naive back-projection. But the filter amplifies high-frequency noise, resulting in the visible texture and artifacts seen in the noisy reconstruction.

### C3 (1.0 pts)

Apply the MLEM reconstruction from task B3 to the noisy sinogram and view how the image evolve for each iteration up until e.g 100 iterations. Visualize a reconstructed slice for different iteration numbers and comment on the results. Discuss how your observations can be explained in terms of overfitting.

In [13]:
iterations = 100
F_mlem_noisy, error_noisy = MLEM(A, G_noisy[:, :, slice_idx], np.array([image_size[0], image_size[0], 1]), iterations)

# YOUR CODE HERE
iters_to_show = [0, 9, 49, 99]
fig, axes = plt.subplots(1, len(iters_to_show), figsize=(16, 4))

for i, iter_idx in enumerate(iters_to_show):
    img_slice = F_mlem_noisy[:, :, 0, iter_idx]
    
    axes[i].imshow(img_slice, cmap='gray_r')
    axes[i].set_title(f'Iteration {iter_idx + 1}')
    axes[i].axis('off')

plt.tight_layout()
plt.show()

plt.figure(figsize=(8, 4))
plt.plot(error_noisy)
plt.title("Sum of Squared Residuals (Noisy Data)")
plt.xlabel("Iteration")
plt.ylabel("Error")
plt.grid(True)
plt.show()



![Saved output from cell 33, figure 1](../assets/pet/noisy-mlem.png)

![Saved output from cell 33, figure 2](../assets/notebook-results/pet-cell-33-figure-1.png)


**Answer:**



Iteration 1: The image is extremely blurry with only vague shapes visible.

Iteration 10: Represents the optimal balance; internal structures are sharp and clearly defined with minimal noise.

Iterations 50-100: Image quality degrades significantly. The smooth regions break down into high-frequency "salt-and-pepper" noise artifacts.

The results demonstrate the semi-convergence of MLEM. The error plot shows the residual error dropping quickly at first while fitting the signal and then flattening out. In later iterations, the algorithm continues to minimize error by fitting the random noise in the sinogram, treating it as true data. This overfitting confirms that running MLEM to convergence on noisy data ruins the image, necessitating early stopping around iteration 10 in this case.

### C4 (1.0 pts)

Fine-tuning of the optimal number of iterations is clearly necessary. For this project we have the noise-free images available meaning that we can see how the reconstructed noisy image deviates from the true image at each iteration. Monitor the sum-of-squared difference between the noisy reconstructed image and the noise-free image reconstructed in B3 at each iteration. 

Select an optimal number of iterations and visualize a slice of the noisy image obtained from the optimal iterative reconstruction, and comment on the difference to the corresponding slice obtained from FBP in task C2. Plot the sum-of-squares error between the non-noisy and noisy images, and make mark the optimal number of iterations in the plot.

In [ ]:
# Compute sum of squares error compared to noise-free reconstruction
sum_squares = np.zeros(iterations)

ref_image = F_mlem[:, :, :, -1]

for i in range(iterations):
    # YOUR CODE HERE
    # Compare to final noise-free iteration
    current_noisy = F_mlem_noisy[:, :, :, i]
    diff = current_noisy - ref_image
    sum_squares[i] = np.sum(diff**2)

# Get optimal iteration index
minidx = np.argmin(sum_squares)
print(f"Optimal number of iterations: {minidx + 1}")
print(f"Minimum SSE: {sum_squares[minidx]:.4f}")

plt.figure(figsize=(10, 5))
plt.plot(range(1, iterations + 1), sum_squares, label='SSE (Noisy vs Noise-free)')
plt.axvline(x=minidx + 1, color='r', linestyle='--', label=f'Optimal: {minidx + 1}')
plt.scatter(minidx + 1, sum_squares[minidx], color='red', zorder=5)

plt.title('Optimization of Iteration Number (Early Stopping)')
plt.xlabel('Iteration')
plt.ylabel('Sum-of-Squared Error')
plt.legend()
plt.grid(True)
plt.show()

fig, ax = plt.subplots(1, 3, figsize=(18, 6))

ax[0].imshow(F_mlem_noisy[:, :, 0, minidx], cmap='gray_r')
ax[0].set_title(f'Optimal MLEM (Iter {minidx + 1})')
ax[0].axis('off')

ax[1].imshow(F_mlem_noisy[:, :, 0, -1], cmap='gray_r')
ax[1].set_title(f'Overfitted MLEM (Iter {iterations})')
ax[1].axis('off')

if 'F_fbp_noisy' in locals():
    if F_fbp_noisy.ndim == 3:
        ax[2].imshow(F_fbp_noisy[:, :, slice_idx], cmap='gray_r')
    else:
        ax[2].imshow(F_fbp_noisy, cmap='gray_r')
    ax[2].set_title('FBP (Noisy)')
    ax[2].axis('off')

plt.tight_layout()
plt.show()

Optimal number of iterations: 8
Minimum SSE: 2967868.2500


![Saved output from cell 37, figure 2](../assets/notebook-results/pet-cell-37-figure-1.png)

![Saved output from cell 37, figure 3](../assets/notebook-results/pet-cell-37-figure-2.png)


**Answer:**



**Optimal Iteration Selection & Visualization**
The optimal number of iterations was determined to be 8. The slice visualized at this iteration presents a smooth, recognizable reconstruction of the phantom, with minimal background noise and clearly defined internal cavities.

* **Noise Reduction:** The MLEM reconstruction is dramatically cleaner. The "Optimal MLEM" image shows smooth gray matter and distinct dark voids, whereas the FBP image is overlaid with intense, grainy, high-frequency noise that gives it a "salt-and-pepper" texture.
* **Artifacts:** FBP exhibits streak artifacts radiating from the center, which are largely absent in the MLEM reconstruction.
* **Clarity:** While FBP preserves edge sharpness well, the noise overwhelms low-contrast features. MLEM, by stopping early, effectively regularizes the image, sacrificing some potential high-frequency edge resolution to preserve the structural integrity of the object against the noise.

### C5 (0.5 pts)

Discuss why the sum-of-squared difference between the sinogram and the forward projection (the error in B3) cannot be used to find an optimal number of iterations given noisy data.

**Answer:**


The sum-of-squared difference between the sinogram and the forward projection (the residual error) cannot be used to find the optimal number of iterations for noisy data because of the nature of the Maximum Likelihood Expectation Maximization (MLEM) algorithm.

Fitting the Noise: As the MLEM algorithm iterates, it attempts to minimize the difference between the measured data (the noisy sinogram) and its estimate (the forward projection). Since the algorithm assumes all variations in the data are "signal," it will eventually start fitting the random noise inherent in the sinogram as if it were real physical structure.

The residual error, the difference between the forward projection and the noisy sinogram will generally continue to decrease or plateau as iterations increase, which indicates a better mathematical fit to the data. However, this "better fit" corresponds to an increasingly degraded image full of high-frequency artifacts, as the reconstruction becomes overfitted to the noise.